# Automated Land Cover Sampling Strategy
### This notebook performs an end-to-end spatial analysis to generate high-quality training samples from Corine Land Cover (CLC) data and Sentinel-2 NDVI imagery, for the year 2018.
This is the V5 (Fixed Distribution): Approximately 200 polygons sampled across all 28 classes including 4 seasons. 

### inputs : 
1. one raster of NDVI, 
2. one cropped shp of CLC or any local reference layer
### outputs: 
1. Square polygon of of pure samples 
2. Randomly selected up to 60 polygons per class

## Step 0: Environment Setup

In [1]:
# 1. Import libraries
import geopandas as gpd
import pandas as pd
import rasterio
from rasterstats import zonal_stats
import folium
from folium import plugins
import json
import ee
import os
from ipyleaflet import Map, DrawControl
import time
import datetime
import math, requests
from pathlib import Path
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML, clear_output, Markdown
import numpy as np
from rasterio.mask import mask
from shapely.geometry import Point, box, mapping
from shapely.ops import unary_union, transform
from shapely.validation import make_valid
from sklearn.ensemble import IsolationForest
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap
import pyproj
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import textwrap
import random
from cdsetool.query import query_features
from cdsetool.download import download_features
from cdsetool.credentials import Credentials
import zipfile
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
import shutil
from shapely.prepared import prep
import calendar
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)

print ("Libraries imported correctly!")

C:\ProgramData\anaconda3\envs\cop_t24_env\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Libraries imported correctly!


In [2]:
# 2. Initialize Earth Engine and Select bounding box AOI

# Initialize (authenticate once with ee.Authenticate() if needed)
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print ("Earth Engine initialized!")

def mask_s2_clouds(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return (
        image.updateMask(mask)
        .divide(10000)
    )

def get_seasonal_ndvi(aoi, start_date, end_date):

    s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                20
            )
        )
        .map(mask_s2_clouds)
    )

    composite = s2.median()

    ndvi = (
        composite.normalizedDifference(
            ["B8", "B4"]
        )
        .rename("NDVI")
    )

    return ndvi


# =========================
# AOI SELECTION
# =========================

mode = input(
    "\nChoose input mode:\n"
    "1 = Enter longitude/latitude\n"
    "2 = Select point on map\n\n"
    "Option: "
).strip()

# -------------------------
# COORDINATE INPUT
# -------------------------
if mode == "1":

    coords = input(
        "\nEnter longitude,latitude "
        "(example Seville: -5.9845,37.3891): "
    )

    POINT_LON, POINT_LAT = map(
        float,
        coords.split(",")
    )

# -------------------------
# MAP CLICK
# -------------------------
elif mode == "2":

    from ipyleaflet import Map, Marker
    from IPython.display import display

    clicked = {}

    m = Map(center=(40, -3), zoom=6)

    marker = Marker(location=(40, -3))
    m.add(marker)

    def handle_click(**kwargs):

        if kwargs.get('type') == 'click':

            latlon = kwargs.get('coordinates')

            marker.location = latlon

            clicked["lat"] = latlon[0]
            clicked["lon"] = latlon[1]

            print(
                f"Selected:"
                f" {latlon[1]:.5f}, {latlon[0]:.5f}"
            )

    m.on_interaction(handle_click)

    display(m)

    input("\nClick map then press ENTER here...")

    POINT_LAT = clicked["lat"]
    POINT_LON = clicked["lon"]

else:

    raise ValueError("Invalid option")

# -------------------------
# SMALL AOI AROUND POINT
# -------------------------
BUFFER_DEG = 0.15

geom_4326 = box(
    POINT_LON - BUFFER_DEG,
    POINT_LAT - BUFFER_DEG,
    POINT_LON + BUFFER_DEG,
    POINT_LAT + BUFFER_DEG
)

geom_wkt = geom_4326.wkt

# -------------------------
# AUTOMATIC TILE DETECTION
# -------------------------
point = ee.Geometry.Point([
    POINT_LON,
    POINT_LAT
])

s2 = (
    ee.ImageCollection(
        "COPERNICUS/S2_SR_HARMONIZED"
    )
    .filterBounds(point)
    .first()
)

TARGET_TILE = s2.get(
    "MGRS_TILE"
).getInfo()

TARGET_YEAR = int(input("Enter processing year (example 2025): ").strip())

print("✅ Using tile:", TARGET_TILE)
print("✅ Using year:", TARGET_YEAR)

Earth Engine initialized!



Choose input mode:
1 = Enter longitude/latitude
2 = Select point on map

Option:  1

Enter longitude,latitude (example Seville: -5.9845,37.3891):  -5.9845,37.3891
Enter processing year (example 2025):  2018


✅ Using tile: 29SQB
✅ Using year: 2018


In [23]:
# 3. Setup Paths
root_dir = r'C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN'
#input_shp = os.path.join(root_dir, 'CLC_L3_2018_spain.shp')
input_gpkg = r"C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN\reclass_CLC_28class.gpkg"
CLC_LAYER = "reclassCLCin28Class"
raster_path = os.path.join(root_dir, 'NDVI_Tile_29SQB_2018_spain.tif')
SEASON_OUTPUT = os.path.join(
    root_dir,
    "seasonal_outputs"
)

# Define sub-folders
folders = ['split_outputs', 'processed_poly', 'merged_reclassified', 'exports']
paths = {folder: os.path.join(root_dir, folder) for folder in folders}
SOURCE_CLASS_FIELD = "clc_l3"   # change if your modified classes are stored in another field
TOP_N_HIGH_AREA = 8              # choose how many highest-area classes to keep
FORCE_INCLUDE = {"310", "325", "330", "331"}
OTHER_CLASS = "999"

# Create all necessary directories
for path in paths.values():
    if not os.path.exists(path):
        os.makedirs(path)

print("Libraries imported and directories initialized.")

Libraries imported and directories initialized.


## Step 1: Reclassification and Class Splitting

We simplify the CLC Level 3 codes into broader categories (Level 2) where necessary, then split the dataset into individual files based on the class.

In [24]:
# 4. Change the values for reclassification based on the need
def log_time(msg, start=None):
    now = time.time()
    ts = datetime.now().strftime("%H:%M:%S")
    if start:
        print(f"[{ts}] {msg} — {(now - start):.1f}s")
    else:
        print(f"[{ts}] {msg}")
    return now

# ----------------------------
# Status system (like flood example)
# ----------------------------
VERBOSE = True
MIN_DISPLAY_SEC = 1
_last_status_time = None

def _wait_for_previous(min_seconds: float):
    global _last_status_time
    if _last_status_time is not None:
        elapsed = time.time() - _last_status_time
        if elapsed < min_seconds:
            time.sleep(min_seconds - elapsed)

def status(msg: str, *, done: bool=False, min_seconds: float = MIN_DISPLAY_SEC):
    if not VERBOSE:
        return
    _wait_for_previous(min_seconds)
    with status_area:
        status_area.clear_output(wait=True)
        style = "color:#333;"
        if done:
            style = "color:#2e7d32;"
        display(HTML(
            f'<div style="font-family:Segoe UI,Arial;font-size:14px;'
            f'padding:6px 8px;border-left:4px solid #888;{style}">{msg}</div>'
        ))
    global _last_status_time
    _last_status_time = time.time()

class StepTimer:
    def __init__(self, start_msg: str, done_label: str = "done", min_seconds: float = MIN_DISPLAY_SEC):
        self.start_msg = start_msg
        self.done_label = done_label
        self.min_seconds = min_seconds
        self.t0 = None
    def __enter__(self):
        self.t0 = time.time()
        status(self.start_msg, done=False, min_seconds=self.min_seconds)
        return self
    def __exit__(self, exc_type, exc, tb):
        dt = time.time() - self.t0
        if exc_type is None:
            status(f"{self.start_msg} — {self.done_label} in {dt:.2f}s", done=True, min_seconds=self.min_seconds)
        else:
            with status_area:
                status_area.clear_output(wait=True)
                display(HTML(
                    f'<div style="font-family:Segoe UI,Arial;font-size:14px;'
                    f'padding:6px 8px;border-left:4px solid #c62828; color:#c62828;">'
                    f'⚠️ Error during: {self.start_msg}<br><code>{exc}</code></div>'
                ))

# Dedicated output area
status_area = widgets.Output()
display(status_area)


with StepTimer("Reading and building high-coverage targeted classes..."):
    gdf_main = gpd.read_file(input_gpkg, layer=CLC_LAYER)

    # -----------------------------------------
    # 1. Base class from source field
    # -----------------------------------------
    gdf_main["base_class"] = gdf_main[SOURCE_CLASS_FIELD].astype(str)
    
    # -----------------------------------------
    # 2. Compute area per class in equal-area CRS
    # -----------------------------------------
    gdf_area = gdf_main.copy()
    if gdf_area.crs is None:
        raise ValueError("❌ Input CLC layer has no CRS.")
    
    if gdf_area.crs.to_string() != "EPSG:3035":
        gdf_area = gdf_area.to_crs("EPSG:3035")
    
    gdf_area["area_m2"] = gdf_area.geometry.area
    
    area_by_class = (
        gdf_area
        .groupby("base_class", as_index=False)["area_m2"]
        .sum()
        .sort_values("area_m2", ascending=False)
    )
    
    # -----------------------------------------
    # 3. Choose highest-area classes + forced classes
    # -----------------------------------------
    top_classes = (
        area_by_class.loc[
            ~area_by_class["base_class"].isin(FORCE_INCLUDE),
            "base_class"
        ]
        .head(TOP_N_HIGH_AREA)
        .tolist()
    )
    
    keep_classes = set(top_classes) | FORCE_INCLUDE
    
    print("✅ Highest-area classes kept:", sorted(top_classes))
    print("✅ Forced classes included:", sorted(FORCE_INCLUDE))
    print("✅ Final kept classes:", sorted(keep_classes))
    
    # -----------------------------------------
    # 4. Group all others into 999
    # -----------------------------------------
    gdf_main["clc_l3"] = gdf_main["base_class"].where(
        gdf_main["base_class"].isin(keep_classes),
        OTHER_CLASS
    )

    # -----------------------------------------
    # 5. Save split outputs
    # -----------------------------------------
    split_dir = paths["split_outputs"]
    
    # optional cleanup to avoid mixing old files
    for f in os.listdir(split_dir):
        if f.endswith(".gpkg") or f.endswith(".shp"):
            try:
                os.remove(os.path.join(split_dir, f))
            except:
                pass
    
    for cl, g in gdf_main.groupby("clc_l3"):
        out_file = os.path.join(split_dir, f"CLC_Class_{cl}.gpkg")
        g.to_file(out_file, driver="GPKG")
    
    print("✅ Split classes saved to:", split_dir)
    print("Classes written:", sorted(gdf_main["clc_l3"].unique()))

Output()

✅ Highest-area classes kept: ['211', '212', '213', '244', '311', '312', '323', '324']
✅ Forced classes included: ['310', '325', '330', '331']
✅ Final kept classes: ['211', '212', '213', '244', '310', '311', '312', '323', '324', '325', '330', '331']
✅ Split classes saved to: C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN\split_outputs
Classes written: ['211', '212', '213', '244', '310', '311', '312', '323', '324', '325', '330', '331', '999']


## Step 2: Spatial Filtering (Inner Buffering & Square Creation)

To ensure spectral purity, we:

Apply a -100m inner buffer to avoid "edge effects" near class boundaries.

Generate a 40x40m square (20m radius) training site at the centroid of the safe zone.


In [25]:
# 5. Change the inner buffer distance as per need
# Dedicated output area
status_area = widgets.Output()
display(status_area)

INNER_BUFFER_DIST = -100 #in m
SQUARE_HALF_SIDE = 20  #one side distance from point, total buffer distance 40m
AREA_FILTER = 100 #change the area filter to remove tiny polygons

all_files = [f for f in os.listdir(paths['split_outputs']) if f.endswith(('.shp', '.gpkg'))]

with StepTimer("Applying inner buffer and creating squares..."):
    for file_name in all_files:
        gdf = gpd.read_file(os.path.join(paths['split_outputs'], file_name))
        if gdf.empty: continue
        if gdf.crs.is_geographic: gdf = gdf.to_crs("EPSG:3035")
        
        # 100m Inner Buffer
        gdf['inner_geom'] = gdf.geometry.buffer(INNER_BUFFER_DIST)
        gdf = gdf[~gdf['inner_geom'].is_empty & (gdf['inner_geom'].area > AREA_FILTER)].copy()  #change the area here
    
        if not gdf.empty:
            # Create Square Site
            gdf['centroid_pt'] = gdf['inner_geom'].centroid
            gdf['square_geom'] = gdf['centroid_pt'].buffer(SQUARE_HALF_SIDE, cap_style=3)
            
            # Keep only sites fully within the safe zone
            within_mask = gdf.apply(lambda row: row['square_geom'].within(row['inner_geom']), axis=1)
            gdf_final = gdf[within_mask].set_geometry('square_geom').drop(columns=['inner_geom', 'centroid_pt', 'geometry'])
            gdf_final = gdf_final.rename_geometry('geometry')
    
            gdf_final.to_file(os.path.join(paths['processed_poly'], f"processed_{file_name}"))
        print(f"Processed: {file_name} ({len(gdf_final)} features)")

Output()

Processed: CLC_Class_211.gpkg (210 features)
Processed: CLC_Class_212.gpkg (195 features)
Processed: CLC_Class_213.gpkg (17 features)
Processed: CLC_Class_244.gpkg (97 features)
Processed: CLC_Class_310.gpkg (414 features)
Processed: CLC_Class_311.gpkg (85 features)
Processed: CLC_Class_312.gpkg (78 features)
Processed: CLC_Class_323.gpkg (248 features)
Processed: CLC_Class_324.gpkg (104 features)
Processed: CLC_Class_325.gpkg (11 features)
Processed: CLC_Class_330.gpkg (22 features)
Processed: CLC_Class_331.gpkg (0 features)
Processed: CLC_Class_999.gpkg (819 features)


## Step 3: Merging Processed Datasets
We merge all individual class files back into a single GeoPackage for final spectral analysis.

In [26]:
# 6. Merging Processed Datasets
status_area = widgets.Output()
display(status_area)
with StepTimer("Merging datasets..."):
    merge_files = [f for f in os.listdir(paths['processed_poly']) if f.endswith('.gpkg')]
    gdf_list = [gpd.read_file(os.path.join(paths['processed_poly'], f)) for f in merge_files]
    
    if gdf_list:
        merged_gdf = pd.concat(gdf_list, ignore_index=True)
        merged_gdf = gpd.GeoDataFrame(merged_gdf, crs=gdf_list[0].crs)
        merged_path = os.path.join(paths['merged_reclassified'], "Merged_CLC_reclassified_Classes.gpkg")
        merged_gdf.to_file(merged_path, driver="GPKG")
        print(f"Successfully merged {len(gdf_list)} files.")

Output()

Successfully merged 13 files.


In [28]:
# 7. Feature extraction + AI-assisted ranking + NDVI ----

# =========================
# CONFIG
# =========================
USERNAME = "francisco.domingues@uab.es"
PASSWORD = "19@Greenday81"

credentials = Credentials(USERNAME, PASSWORD)

export_dir = os.path.join("C:\\Users\\fdomingues\\cop_t24\\CLC_sampling_process_SPAIN", "exports")
os.makedirs(export_dir, exist_ok=True)

winter_end_day = 29 if calendar.isleap(TARGET_YEAR) else 28

SEASONS = {
    "winter": (f"{TARGET_YEAR-1}-12-01", f"{TARGET_YEAR}-02-{winter_end_day:02d}"),
    "spring": (f"{TARGET_YEAR}-03-01", f"{TARGET_YEAR}-05-31"),
    "summer": (f"{TARGET_YEAR}-06-01", f"{TARGET_YEAR}-08-31"),
    "autumn": (f"{TARGET_YEAR}-09-01", f"{TARGET_YEAR}-11-30")
}

# AOI already defined in Cell 2
print("✅ Using tile:", TARGET_TILE)

# =========================
# FUNCTION: compute NDVI
# =========================
def compute_ndvi_array(b4_path, b8_path, scl_path, season):
    
    with rasterio.open(b4_path) as red:
        red_arr = red.read(1).astype(np.float32)
        profile = red.profile.copy()
        transform = red.transform

    with rasterio.open(b8_path) as nir:
        nir_arr = nir.read(1).astype(np.float32)

    with rasterio.open(scl_path) as scl_src:
        scl_arr = scl_src.read(1)

        # ✅ simple upscaling (20m → 10m)
        scale = int(red_arr.shape[0] / scl_arr.shape[0])
        scl_resampled = scl_arr.repeat(scale, axis=0).repeat(scale, axis=1)
        scl_resampled = scl_resampled[:red_arr.shape[0], :red_arr.shape[1]]

    # cloud mask
    if season == "winter":
    
        # less aggressive in winter
        cloud_mask = np.isin(
            scl_resampled,
            [3, 9, 10]
        )
    
    else:
    
        cloud_mask = np.isin(
            scl_resampled,
            [3, 8, 9, 10]
        )
        
    ndvi = (nir_arr - red_arr) / (nir_arr + red_arr + 1e-6)
    ndvi[cloud_mask] = np.nan

    return ndvi, profile
# =========================
# MAIN LOOP
# =========================
    
MAX_SCENES = 16   # try 5–10
used_scenes_log = []
season_scene_dates = {}

for season, (start_date, end_date) in SEASONS.items():

    if season == "w":
        print(f"\n🔍 Processing {season}...")
    else:
        continue
        
    # -------------------------
    # Query Sentinel-2
    # -------------------------
    features = list(query_features(
        "SENTINEL-2",
        {
            "contentDateStartGe": start_date,
            "contentDateStartLe": end_date,
            "productType": "S2MSI2A",
            "cloudCover": [0, 40],
            "geometry": geom_wkt
        }
    ))

    if len(features) == 0:
        print(f"❌ No data for {season}")
        continue

    # sort by cloud cover
    def get_cloud_cover(f):
        try:
            return f.get("cloudCover", 100)
        except:
            return 100
    
    
    features_sorted = sorted(features, key=get_cloud_cover)
    features_sorted = features_sorted[:MAX_SCENES]

    # keep only scenes from that tile
    candidates = features_sorted    
    
    ndvi_stack = []
    profile_ref = None

    # -------------------------
    # PROCESS EACH SCENE
    # -------------------------
    for i, candidate in enumerate(candidates):

        print(f"   ↳ Scene {i+1}/{len(candidates)}")
        scene_start = time.time()
        out_dir = os.path.join(export_dir, f"{season}_tmp_{i}")
        os.makedirs(out_dir, exist_ok=True)
        
        t0 = time.time()

        list(download_features(
            [candidate],
            out_dir,
            {"credentials": credentials}
        ))

        print(
            f"Download took "
            f"{time.time()-t0:.1f} sec"
        )    

        t0 = time.time()

        # extract ZIP
        for f in os.listdir(out_dir):
            if f.endswith(".zip"):
                zip_path = os.path.join(out_dir, f)
                
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(out_dir)
                
                # ✅ DELETE zip immediately
                os.remove(zip_path)

        print(
            f"Unzip took "
            f"{time.time()-t0:.1f} sec"
        )

        # find SAFE folder
        safe_dirs = [
            os.path.join(out_dir, d)
            for d in os.listdir(out_dir)
            if d.endswith(".SAFE")
        ]

        if not safe_dirs:
            continue

        safe_dir = safe_dirs[0]
        
        safe_name = os.path.basename(safe_dir)
        
        safe_tile = safe_name.split("_")[5]

        acq_token = safe_name.split("_")[2]   # e.g. 20241203T110419
        acq_date = f"{acq_token[:4]}-{acq_token[4:6]}-{acq_token[6:8]}"
        acq_datetime = (
            f"{acq_token[:4]}-{acq_token[4:6]}-{acq_token[6:8]} "
            f"{acq_token[9:11]}:{acq_token[11:13]}:{acq_token[13:15]}"
        )

        print(f"   SAFE tile: {safe_tile}")
        
        if safe_tile != f"T{TARGET_TILE}":
        
            print("   ⚠️ Wrong tile → skipping")
        
            continue     
            
        
        # find bands
        b4 = b8 = scl = None

        for root, _, files in os.walk(safe_dir):
            for f in files:
                full_path = os.path.join(root, f)

                if "R10m" in root:
                    if "_B04_10m.jp2" in f:
                        b4 = full_path
                    elif "_B08_10m.jp2" in f:
                        b8 = full_path

                if "R20m" in root and "_SCL_20m.jp2" in f:
                    scl = full_path

        if not (b4 and b8 and scl):
            print("   ❌ Missing bands → skipping")
            continue

        # NDVI
        ndvi, profile = compute_ndvi_array(
            b4,
            b8,
            scl,
            season
        )

        print(
            "NDVI:",
            round(time.time()-scene_start,1),
            "sec"
        )

        if np.isnan(ndvi).all():
            print("   ❌ all pixels masked")
            continue

        ndvi_stack.append(ndvi)
        
        used_scenes_log.append({
            "season": season,
            "tile": safe_tile,
            "safe_name": safe_name,
            "acquisition_date": acq_date,
            "acquisition_datetime": acq_datetime
        })
        
        season_scene_dates.setdefault(season, [])
        season_scene_dates[season].append(acq_date)

        if profile_ref is None:
            profile_ref = profile

        print(f"   ✅ valid pixels: {np.sum(np.isfinite(ndvi))/ndvi.size:.2f}")

    # -------------------------
    # COMPOSITE
    # -------------------------
    if len(ndvi_stack) == 0:
        print(f"❌ No valid NDVI for {season}")
        continue

    print(f"✅ Combining {len(ndvi_stack)} scenes...")

    stack = np.stack(ndvi_stack)
    ndvi_median = np.nanmedian(stack, axis=0)

    ndvi_out = os.path.join(export_dir, f"ndvi_{season}.tif")

    profile_ref.update(
        driver="GTiff",
        dtype=rasterio.float32,
        count=1,
        nodata=-9999,
        compress="lzw"
    )

    ndvi_median[np.isnan(ndvi_median)] = -9999

    with rasterio.open(ndvi_out, 'w', **profile_ref) as dst:
        dst.write(ndvi_median.astype(np.float32), 1)

    print(f"✅ FINAL NDVI exported: {ndvi_out}")
    print("🧹 Cleaning temporary files...")

    for i in range(MAX_SCENES):
        tmp_folder = os.path.join(export_dir, f"{season}_tmp_{i}")
    
        if os.path.exists(tmp_folder):
            shutil.rmtree(tmp_folder, ignore_errors=True)
            print(f"   ✅ Deleted: {tmp_folder}")


scene_log_df = pd.DataFrame(used_scenes_log)

if not scene_log_df.empty:
    scene_log_df = scene_log_df.sort_values(["season", "acquisition_datetime"])
    scene_log_csv = os.path.join(export_dir, "used_s2_scenes_by_season.csv")
    scene_log_df.to_csv(scene_log_csv, index=False)
    print("Saved used scene log:")
    print(scene_log_csv)


# Paths from your script
#root_dir = r'C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN'  # keep as in your file
#raster_ndvi_path = os.path.join(root_dir, 'NDVI_Tile_29SQB_2018_spain.tif')  # NDVI raster (your script)  # ⟵ uses your NDVI input
merged_path = os.path.join(root_dir, 'merged_reclassified', 'Merged_CLC_reclassified_Classes.gpkg')  # ⟵ your merge output
#export_dir = os.path.join(root_dir, 'exports')

# OPTIONAL: local multi-band Sentinel-2 stack (same CRS/resolution as NDVI)
# If you have a 6-band stack (B2,B3,B4,B8,B11,B12), set this path; else leave as None.
raster_s2_stack_path = None  # e.g., os.path.join(root_dir, 'S2_L2A_2018_stack.tif')

# Quality scoring weights (tune if you like)
W = {
    "ndvi_std": 0.35,          # lower is better
    "ndvi_range": 0.15,        # narrower p90-p10 is better
    "s2_band_std": 0.25,       # lower multi-band std is better
    "interior_margin": 0.15,   # more distance to boundary is better
    "iso_forest": 0.10         # anomaly score -> lower is better
}

# Distance threshold for spatial dispersion (meters)
MIN_SPACING = 500

# Default purity gates (adaptive top-up will relax these)
STDEV_LIMITS = [0.06, 0.08, 0.10]  # your original 0.06 first, then looser
#INNER_BUFFERS = [-100, -60, -40, -20]  # meters
#SQUARE_HALF_SIDES = [20, 16, 12, 10]   # meters -> squares 40, 32, 24, 20 m

INNER_BUFFERS = [-60, -40, -20, -10, 0]   # added 0 as a last-resort fallback
SQUARE_HALF_SIDES = [16, 12, 10, 8]       # 32m, 24m, 20m, 16m squares

# Helper: normalize feature to 0..1 within each class (robust to outliers)
def robust_minmax(x):
    q1, q99 = np.nanpercentile(x, [1, 99])
    return np.clip((x - q1) / max(1e-9, (q99 - q1)), 0, 1)

# Greedy spacing filter
def enforce_min_spacing(gdf, min_dist):
    selected = []
    tree = None
    try:
        from shapely.strtree import STRtree
        tree = STRtree(gdf.geometry.values)
    except Exception:
        # fallback without spatial index (slower)
        pass

    for i, geom in enumerate(gdf.geometry.values):
        if not selected:
            selected.append(i)
            continue
        ok = True
        for j in selected:
            if geom.distance(gdf.geometry.values[j]) < min_dist:
                ok = False
                break
        if ok:
            selected.append(i)
    return gdf.iloc[selected]
print("Finishing setting rankings!")

✅ Using tile: 29SQB
Finishing setting rankings!


## Step 4: Re‑generate candidate squares adaptively (per class)

We’ll rebuild the candidate squares inside your merged file, trying multiple inner buffers and square sizes so that narrow polygons still get a chance. (replaces the single –100 m buffer + 40 m square.)

In [29]:
# 8. Re‑generate candidate squares adaptively (per class) ----
# REQUIREMENTS: StepTimer, status(), INNER_BUFFERS, SQUARE_HALF_SIDES, merged_path must exist

status_area = widgets.Output()
display(status_area)

with StepTimer("Re‑generating candidate squares adaptively (per class)..."):

    # Uses the paths you already defined in your notebook:
    # input_shp, raster_path
    
    assert os.path.exists(input_gpkg), f"CLC not found: {input_gpkg}"
    assert os.path.exists(raster_path), f"NDVI not found: {raster_path}"
    
    gdf_src0 = gpd.read_file(input_gpkg, layer=CLC_LAYER)
    #print("CLC: features =", len(gdf_src0), "CRS =", gdf_src0.crs)
    
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        r_bounds = src.bounds
        r_bbox = box(*r_bounds)
    #print("NDVI: CRS =", r_crs, "bounds =", tuple(round(v,3) for v in r_bounds))
    
    # Reproject CLC to the raster CRS to test intersection with the NDVI coverage
    gdf_src_r = gdf_src0.to_crs(r_crs)
    intersects = gdf_src_r.intersects(r_bbox)
    print("CLC polygons intersecting the NDVI extent:", int(intersects.sum()))
    if intersects.sum() == 0:
        raise RuntimeError("Your NDVI raster does not cover the CLC area (or CRS mismatch). Check the NDVI tile and AOI.")
    
    # 1) Read CLC and add clc_l3 if missing
    gdf_clc = gpd.read_file(input_gpkg, layer=CLC_LAYER)
    if 'clc_l3' not in gdf_clc.columns:
        source_code_col = 'Code_18' if 'Code_18' in gdf_clc.columns else None
        if not source_code_col:
            raise KeyError("Could not find the CLC code column (e.g., 'Code_18').")
        gdf_clc['clc_l3'] = gdf_clc[source_code_col].apply(reclassify_clc)
    gdf_clc = gdf_clc.dropna(subset=['clc_l3']).copy()
    
    # 2) Clip to the NDVI raster extent (in the raster CRS)
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        r_bbox = box(*src.bounds)
    
    gdf_clc_r = gdf_clc.to_crs(r_crs)
    gdf_clc_r = gdf_clc_r[gdf_clc_r.intersects(r_bbox)].copy()
    if gdf_clc_r.empty:
        raise RuntimeError("After clipping to NDVI extent, no CLC polygons remain. Verify the NDVI tile and your AOI.")
    
    # 3) Project to a metric CRS for buffering (EPSG:3035 is good for Europe)
    gdf_clc_m = gdf_clc_r.to_crs("EPSG:3035")
    
    print(f"CLC (clipped to NDVI): {len(gdf_clc_m)} polygons across {gdf_clc_m['clc_l3'].nunique()} classes.")
    # 0) Read and sanity-check merged polygons
    if not os.path.exists(merged_path):
        raise FileNotFoundError(f"Merged file not found at:\n{merged_path}\nRun the merge step first.")

    merged_gdf = gpd.read_file(merged_path)  # must contain 'clc_l3'
    if merged_gdf.empty:
        raise RuntimeError("Merged dataset is empty. Step 2 likely removed everything—relax buffers/square size.")

    if "clc_l3" not in merged_gdf.columns:
        raise KeyError("Column 'clc_l3' not found in merged_gdf. Ensure reclassification step preserved it.")

    def make_square_from_point(pt, half_side):
        x, y = pt.x, pt.y
        return box(x - half_side, y - half_side, x + half_side, y + half_side)

    def make_candidates_for_class(gdf_class, inner_buffers, half_sides):
        rows = []
        gdfc = gdf_class.copy()
        # Heal invalid geometries to avoid errors in negative buffer
        gdfc['geometry'] = gdfc.geometry.buffer(0)

        for ib in inner_buffers:
            safe = gdfc.copy()
            safe["safe_geom"] = safe.geometry.buffer(ib)
            safe = safe[(~safe["safe_geom"].is_empty) & (safe["safe_geom"].is_valid)].copy()
            if safe.empty:
                continue

            cent = safe["safe_geom"].centroid

            for hs in half_sides:
                squares = cent.apply(lambda p: make_square_from_point(p, hs))
                # Keep only squares fully within the safe area
                mask = [sq.within(sg) for sq, sg in zip(squares.values, safe["safe_geom"].values)]
                if not any(mask):
                    continue

                safe2 = safe.loc[mask].copy()
                safe2["geometry"]     = squares.loc[safe2.index].values
                safe2["ib_m"]         = ib
                safe2["half_side_m"]  = hs
                safe2 = safe2.drop(columns=["safe_geom"])
                rows.append(safe2)

        if rows:
            return pd.concat(rows, ignore_index=True)
        # Return empty GeoDataFrame with expected schema
        return gpd.GeoDataFrame(columns=list(gdf_class.columns)+["ib_m","half_side_m"],
                                geometry="geometry", crs=gdf_class.crs)

    candidates = []
    report = []
    for cl, g in gdf_clc_m.groupby("clc_l3"):
        c = make_candidates_for_class(g, INNER_BUFFERS, SQUARE_HALF_SIDES)
        n = 0 if c is None or c.empty else len(c)
        report.append((cl, n))
        if n > 0:
            if "clc_l3" not in c.columns:
                c = c.assign(clc_l3=cl)
            candidates.append(c)

    if not candidates:
        msg = "\n".join([f"  - Class {cl}: {n} candidates" for cl, n in report])
        raise RuntimeError("Still no candidates. Try INNER_BUFFERS=[-40,-20,-10,0] and SQUARE_HALF_SIDES=[12,10,8,6].\n" + msg)

    candidates_gdf = gpd.GeoDataFrame(pd.concat(candidates, ignore_index=True), crs=gdf_clc_m.crs)



    
    total = len(candidates_gdf)
    status(f"✅ Candidate squares generated: {total}")
    print("Per-class candidate counts:")
    display(candidates_gdf["clc_l3"].value_counts().sort_index())
    

Output()

CLC polygons intersecting the NDVI extent: 111233
CLC (clipped to NDVI): 111233 polygons across 28 classes.
Per-class candidate counts:


clc_l3
11       3533
12       2200
13       1365
14        580
211      5010
212      4636
213       340
221       621
222      2262
223      2376
231      2925
24       3197
244      3204
310    169023
311      3082
312      2181
313       791
321      1873
323      8606
324      3042
325      5067
330       612
331       162
411       149
421        20
422        20
51        836
52         20
Name: count, dtype: int64

## Feature extraction (NDVI + optional S2)

We compute NDVI stats using your NDVI raster; if you provide a local S2 band stack, we also compute per‑band standard deviations for additional purity signals.

In [ ]:
# 9. Feature extraction (NDVI + optional S2)  — REVISED, robust
status_area = widgets.Output()
display(status_area)

SHRINK_FACTORS = [1.0, 0.9, 0.8, 0.7, 0.6]

# =========================================================
# SEASONAL NDVI INPUTS
# =========================================================
export_dir = os.path.join("C:\\Users\\fdomingues\\cop_t24\\CLC_sampling_process_SPAIN", "exports")

SEASONAL_NDVI = {
    "winter": os.path.join(export_dir, "ndvi_winter.tif"),
    "spring": os.path.join(export_dir, "ndvi_spring.tif"),
    "summer": os.path.join(export_dir, "ndvi_summer.tif"),
    "autumn": os.path.join(export_dir, "ndvi_autumn.tif")
}

all_season_results = []

used_grid_ids = set()   # ✅ global memory across all seasons
GRID_SIZE = 200         # meters

def get_grid_id(geom, size=200):
    c = geom.centroid
    return (int(c.x // size), int(c.y // size))

# --------------------------------------------------------------
# GEOMETRY CLEANING HELPERS
# --------------------------------------------------------------
def clean_geom(geom):
    if geom is None or geom.is_empty:
        return None

    try:
        geom = make_valid(geom)
    except Exception:
        try:
            geom = geom.buffer(0)
        except Exception:
            return None

    if geom is None or geom.is_empty:
        return None

    # Keep only polygonal parts
    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ["Polygon", "MultiPolygon"]]
        if len(polys) == 0:
            return None
        geom = unary_union(polys)

    if geom.geom_type not in ["Polygon", "MultiPolygon"]:
        return None

    try:
        geom = geom.buffer(0)
    except Exception:
        pass

    if geom.is_empty:
        return None

    return geom


def clean_gdf_polygons(gdf):
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.apply(clean_geom)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = gdf.explode(index_parts=False)
    return gdf


def safe_union(geoms):
    geoms = [clean_geom(g) for g in geoms]
    geoms = [g for g in geoms if g is not None and not g.is_empty]
    if len(geoms) == 0:
        return None
    try:
        return unary_union(geoms)
    except Exception:
        try:
            return gpd.GeoSeries(geoms).buffer(0).unary_union
        except Exception:
            return None
            
with StepTimer("Extracting features (NDVI)..."):

    for season_name, raster_ndvi_path in SEASONAL_NDVI.items():

        #if season_name.upper() == "WINTER":
        print("\n" + "="*60)
        print(f"PROCESSING SEASON: {season_name.upper()}")
        print("="*60)

        # ================================================================
        # STEP 4 — OPTIMIZED NDVI FEATURE EXTRACTION & SAMPLE SELECTION
        # ================================================================
        
        # --------------------------------------------------------------
        # 4.0 LOAD CANDIDATES FROM PREVIOUS STEP
        # --------------------------------------------------------------
        candidates_gdf = clean_gdf_polygons(candidates_gdf)
        print("Initial candidates:", len(candidates_gdf))
        
        
        # --------------------------------------------------------------
        # 4.1 REDUCE POLYGONS BEFORE NDVI (CRITICAL FOR SPEED)
        # --------------------------------------------------------------
        filtered = []
        for cl, g in candidates_gdf.groupby("clc_l3"):
            if len(g) > 800:       # limit oversampled classes
                g = g.sample(800, random_state=42)
            filtered.append(g)
        
        candidates_gdf = gpd.GeoDataFrame(pd.concat(filtered), crs=candidates_gdf.crs)
        print("Reduced candidates:", len(candidates_gdf))
        
        
        # --------------------------------------------------------------
        # 4.2 FAST NDVI MEAN & STD USING zonal_stats
        # --------------------------------------------------------------
    
        # Reproject to NDVI CRS    
        with rasterio.open(raster_ndvi_path) as src:
            ndvi_crs = src.crs

        # ✅ FIX: fallback if CRS missing
        if ndvi_crs is None:
            print("⚠️ NDVI raster has no CRS — assigning EPSG:3035")
            ndvi_crs = "EPSG:3035"
        
        # ✅ FIX: ensure candidates have CRS
        if candidates_gdf.crs is None:
            print("⚠️ candidates_gdf has no CRS — assigning EPSG:3035")
            candidates_gdf = candidates_gdf.set_crs("EPSG:3035")

        candidates_ndvi = candidates_gdf.to_crs(ndvi_crs)    
        
        if candidates_gdf.crs is None:
            raise ValueError("❌ candidates_gdf has no CRS. You must assign it earlier.")        
        print("Candidates CRS:", candidates_gdf.crs)
        print("NDVI CRS:", ndvi_crs)

        # Extract fast features
        with rasterio.open(raster_ndvi_path) as src2:
            ndvi_nodata = src2.nodata
    
        zs = zonal_stats(
            candidates_ndvi,
            raster_ndvi_path,
            stats=["mean", "std"],
            nodata=ndvi_nodata
        )
        
        mean_std_df = pd.DataFrame(zs)
        mean_std_df.rename(columns={"mean": "ndvi_mean", "std": "ndvi_std"}, inplace=True)
    
        print(mean_std_df.describe())
        
        feat_df = candidates_gdf.reset_index(drop=True).copy()
        feat_df["ndvi_mean"] = mean_std_df["ndvi_mean"]
        feat_df["ndvi_std"]  = mean_std_df["ndvi_std"]
        
        print("NDVI mean/std extracted.")
        
        
        # --------------------------------------------------------------
        # 4.3 SELECT FINALISTS FOR DEEP ANALYSIS
        #     (smaller set -> fast percentiles)
        # --------------------------------------------------------------
        FINALIST_LIMIT = 250     # good balance: purity + speed
        
        finalists = []
        for cl, g in feat_df.groupby("clc_l3"):
            g = g.sort_values("ndvi_std", ascending=True)
            finalists.append(g.head(min(FINALIST_LIMIT, len(g))))
        
        
        finalists = gpd.GeoDataFrame(pd.concat(finalists), crs=feat_df.crs)
        finalists = clean_gdf_polygons(finalists)
        print("Finalists:", len(finalists))

        
        
        # --------------------------------------------------------------
        # 4.4 NDVI PERCENTILES (10, 50, 90) ONLY FOR FINALISTS
        # --------------------------------------------------------------
        t0 = time.time()
        
        def compute_percentiles_single(geom, raster_path):
            with rasterio.open(raster_path) as src:
                try:
                    arr, _ = mask(src, [mapping(geom)], crop=True, filled=False)
                except:
                    return np.nan, np.nan, np.nan
        
                band = arr[0]
        
                vals = band.compressed() if np.ma.isMaskedArray(band) else band.ravel()
                vals = vals[np.isfinite(vals)]
        
                vals = vals[vals > -0.1]   # remove invalid NDVI
        
                if len(vals) < 10:   # important threshold
                    return np.nan, np.nan, np.nan
        
                return (
                    float(np.nanpercentile(vals, 10)),
                    float(np.nanpercentile(vals, 50)),
                    float(np.nanpercentile(vals, 90))
                )
        
        #percentiles = [compute_percentiles_single(g, raster_ndvi_path) for g in finalists.geometry]
        
        finalists_ndvi = finalists.to_crs(ndvi_crs)
        
        percentiles = [
            compute_percentiles_single(g, raster_ndvi_path)
            for g in finalists_ndvi.geometry
        ]    
        
        finalists["ndvi_p10"]   = [p[0] for p in percentiles]
        finalists["ndvi_p50"]   = [p[1] for p in percentiles]
        finalists["ndvi_p90"]   = [p[2] for p in percentiles]
        finalists["ndvi_range"] = finalists["ndvi_p90"] - finalists["ndvi_p10"]
    
        print(f"Percentiles computed.  ⏱️ Took: {time.time() - t0:.2f} sec")
    
        # --- FORCE ALL NDVI COLUMNS TO PURE NUMERIC ---
        num_cols = ["ndvi_mean","ndvi_std","ndvi_p10","ndvi_p50","ndvi_p90","ndvi_range"]
        for c in num_cols:
            finalists[c] = pd.to_numeric(finalists[c], errors="coerce")
            
        # SPEED-UP: simplify original CLC polygons before union (ROBUST)
        gdf_clc_simpl = gdf_clc_m.copy()
        
        # Ensure SAME CRS as finalists (EPSG:3035)
        if gdf_clc_simpl.crs != finalists.crs:
            gdf_clc_simpl = gdf_clc_simpl.to_crs(finalists.crs)
            
        # 1. Fix invalid geometries properly
        gdf_clc_simpl["geometry"] = gdf_clc_simpl["geometry"].apply(make_valid)
        
        # 2. Remove empty geometries
        gdf_clc_simpl = gdf_clc_simpl[~gdf_clc_simpl.is_empty]
        
        # 3. Keep only polygons (avoid GeometryCollections etc.)
        gdf_clc_simpl = gdf_clc_simpl[
            gdf_clc_simpl.geometry.type.isin(["Polygon", "MultiPolygon"])
        ]
        
        # 4. Explode multipolygons (IMPORTANT)
        gdf_clc_simpl = gdf_clc_simpl.explode(index_parts=False)
        
        # 5. Simplify AFTER fixing
        gdf_clc_simpl["geometry"] = gdf_clc_simpl.geometry.simplify(
            50, preserve_topology=True
        )
        
        # 6. Final safety buffer
        gdf_clc_simpl["geometry"] = gdf_clc_simpl["geometry"].buffer(0)
    
        # --------------------------------------------------------------
        # 4.5 INTERIOR MARGIN (distance from square to original CLC)
        # --------------------------------------------------------------
        if finalists.crs.is_geographic:
            finalists = finalists.to_crs("EPSG:3035")
        
        def safe_union(geoms):
            try:
                return unary_union(geoms)
            except Exception:
                # fallback (slower but avoids crash)
                return gpd.GeoSeries(geoms).unary_union
        
        
        gdf_clc_for_union = gdf_clc_m.copy()
        
        # Ensure CRS matches finalists
        if gdf_clc_for_union.crs != finalists.crs:
            gdf_clc_for_union = gdf_clc_for_union.to_crs(finalists.crs)
        
        gdf_clc_for_union = clean_gdf_polygons(gdf_clc_for_union)
        
        orig_union_by_class = {
            cl: safe_union(
                gdf_clc_for_union.loc[gdf_clc_for_union["clc_l3"] == cl, "geometry"]
            )
            for cl in gdf_clc_for_union["clc_l3"].unique()
        }
            
        def interior_margin(row):
            outer = orig_union_by_class.get(row["clc_l3"], None)
            sq = clean_geom(row["geometry"])
        
            if outer is None:
                return 0.0
        
            outer = clean_geom(outer)
        
            if outer is None or outer.is_empty or sq is None:
                return 0.0
        
            try:
                pt = sq.centroid
        
                if not outer.contains(pt):
                    return 0.0
        
                return pt.distance(outer.boundary)
            except Exception:
                return 0.0
            
        finalists["interior_margin_m"] = finalists.apply(interior_margin, axis=1)
        print("Interior margin computed.")
        
        
        # --------------------------------------------------------------
        # 4.6 ISOLATION FOREST (per class)
        # --------------------------------------------------------------
        finalists["iso_penalty"] = np.nan
        
        for cl, g in finalists.groupby("clc_l3"):
            if len(g) < 20:
                finalists.loc[g.index, "iso_penalty"] = 0
                continue
            
            numeric_cols = ["ndvi_mean","ndvi_std","ndvi_range","interior_margin_m"]
            
            X = g[numeric_cols].copy()
            
            # Replace inf with nan
            X = X.replace([np.inf, -np.inf], np.nan)
            
            # Fill only using medians of THESE columns, not all g columns
            X = X.fillna(X.median(numeric_only=True))
            
            iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
            iso.fit(X)
            scores = iso.score_samples(X)
            
            norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
            finalists.loc[g.index, "iso_penalty"] = 1 - norm
        
        print("Isolation Forest done.")
        
        
        # --------------------------------------------------------------
        # 4.7 QUALITY SCORE
        # --------------------------------------------------------------
        def robust_minmax(x):
            q1, q99 = np.nanpercentile(x, [1, 99])
            return np.clip((x - q1) / (q99 - q1 + 1e-9), 0, 1)
        
        W = {"ndvi_std":0.35, "ndvi_range":0.15, "margin":0.25, "iso":0.25}
        
        ranked = []
        for cl, g in finalists.groupby("clc_l3"):
            g = g.copy()
            g["ndvi_std_n"]   = 1 - robust_minmax(g["ndvi_std"])
            g["ndvi_range_n"] = 1 - robust_minmax(g["ndvi_range"])
            g["margin_n"]     =     robust_minmax(g["interior_margin_m"])
            g["iso_n"]        = 1 - robust_minmax(g["iso_penalty"])
            
            g["quality_score"] = (
                W["ndvi_std"]*g["ndvi_std_n"] +
                W["ndvi_range"]*g["ndvi_range_n"] +
                W["margin"]*g["margin_n"] +
                W["iso"]*g["iso_n"]
            ) * 100
            
            g["quality_score"] = g["quality_score"].fillna(0)
            
            ranked.append(g)
        
        ranked = gpd.GeoDataFrame(pd.concat(ranked), crs=finalists.crs)
        ranked = ranked.sort_values(["clc_l3","quality_score"], ascending=[True, False])
        print("Ranking done. Total:", len(ranked))
    
        ranked["sample_source"] = "original"
        
        # --------------------------------------------------------------
        # 4.7b CLASS-ADAPTIVE NDVI TARGETS
        # --------------------------------------------------------------
        class_ndvi_target = {}
        class_ndvi_std = {}
        
        grouped = ranked.groupby("clc_l3")["ndvi_mean"]
        
        for cl, values in grouped:
        
            values = values.dropna()
        
            if len(values) < 10:
                class_ndvi_target[cl] = values.median() if len(values) > 0 else 0
                class_ndvi_std[cl] = values.std() if len(values) > 1 else 0.05
                continue
        
            # remove extremes (robust)
            q_low = values.quantile(0.1)
            q_high = values.quantile(0.9)
            values_f = values[(values >= q_low) & (values <= q_high)]
        
            # class-specific logic
            if cl in [51, 52]:  # water
                target = values_f.quantile(0.25)
            elif cl in [311, 312, 313]:  # forest
                target = values_f.quantile(0.75)
            else:
                target = values_f.median()
        
            class_ndvi_target[cl] = target
            class_ndvi_std[cl] = values_f.std() if len(values_f) > 1 else 0.05
        
        print("NDVI class targets computed.")
        
        # --------------------------------------------------------------
        # 4.8 SELECT TOP 60 PER CLASS WITH ADAPTIVE TOP‑UP
        # --------------------------------------------------------------
        
        TARGET = 60
        MIN_SPACING = 500   # initial spacing requirement
                            
        def enforce_centroid_distance(gdf, min_dist=80):
            selected_idx = []
            selected_centroids = []
        
            for idx, geom in zip(gdf.index, gdf.geometry):
        
                if geom is None or geom.is_empty:
                    continue
        
                c = geom.centroid
                grid_id = get_grid_id(geom, GRID_SIZE)
        
                # ✅ GLOBAL GRID CHECK (VERY IMPORTANT)
                if grid_id in used_grid_ids:
                    continue
        
                keep = True
        
                for sc in selected_centroids:
                    if c.distance(sc) < min_dist:
                        keep = False
                        break
        
                if keep:
                    selected_idx.append(idx)
                    selected_centroids.append(c)
        
                    # ✅ REGISTER IMMEDIATELY
                    used_grid_ids.add(grid_id)
        
            return gdf.loc[selected_idx]
        
        def shrink_geometry(gdf, factor):
            gdf = gdf.copy()
            gdf["geometry"] = gdf.geometry.buffer(0)  # clean
            
            gdf["geometry"] = gdf.geometry.scale(
                xfact=factor,
                yfact=factor,
                origin="center"
            )
            return gdf
            
        def generate_random_samples(polygon, n_samples, size, max_attempts=20000):
            samples = []
            minx, miny, maxx, maxy = polygon.bounds
        
            attempts = 0
        
            while len(samples) < n_samples and attempts < max_attempts:
                attempts += 1
        
                x = random.uniform(minx, maxx)
                y = random.uniform(miny, maxy)
        
                half = size / 2
                sq = box(x - half, y - half, x + half, y + half)
        
                # 🔥 LESS STRICT (critical for 411)
                if not sq.intersects(polygon):
                    continue
        
                samples.append(sq)
        
            return samples
            
        final_rows = []
        
        SQUARE_SIZE = 100  # <-- IMPORTANT: match your real sample size
        
        for cl, g in ranked.groupby("clc_l3"):
    
            t_class_start = time.time()
            
            print(f"\nProcessing class {cl}...")
        
            g_sorted = g.sort_values("quality_score", ascending=False)
        
            best_sel = None
        
            for factor in SHRINK_FACTORS:
        
                if factor < 1.0:
                    g_try = g_sorted.copy()
                    g_try["geometry"] = g_try.geometry.scale(
                        xfact=factor,
                        yfact=factor,
                        origin="center"
                    )
                else:
                    g_try = g_sorted
        
                non_overlap = enforce_centroid_distance(g_try, min_dist=80)
        
                if len(non_overlap) >= TARGET:
                    best_sel = non_overlap.head(TARGET)
                    break
        
                if best_sel is None or len(non_overlap) > len(best_sel):
                    best_sel = non_overlap
        
            # restore ORIGINAL geometries
            sel = g_sorted.loc[best_sel.index].copy()
                                    
            filtered_idx = []
            
            for idx, geom in sel.geometry.items():
            
                if geom is None or geom.is_empty:
                    continue
            
                filtered_idx.append(idx)
            
            sel = sel.loc[filtered_idx].copy()
            
            # ------------------------------------------------------------
            # 🔥 DETERMINISTIC NDVI-OPTIMAL GRID SELECTION (REPLACEMENT)
            # ------------------------------------------------------------
            if len(sel) < TARGET:
            
                needed = TARGET - len(sel)
                print(f"  → Filling {needed} samples using NDVI-optimized grid...")
            
                class_polygon = orig_union_by_class[cl]
                
                # 🔥 shrink polygon so squares fit fully inside
                # ------------------------------------------------------------
                # 🔥 ADAPTIVE SAFE POLYGON
                # ------------------------------------------------------------
                if cl == 411:
                    SAFE_MARGIN = SQUARE_SIZE * 0.25   # MUCH less aggressive
                else:
                    SAFE_MARGIN = SQUARE_SIZE / 2
                
                class_polygon = clean_geom(orig_union_by_class.get(cl, None))
                
                if class_polygon is None or class_polygon.is_empty:
                    print("  ⚠️ Invalid/empty class polygon")
                    final_rows.append(sel.head(TARGET))
                    continue
                
                try:
                    safe_polygon = clean_geom(class_polygon.buffer(-SAFE_MARGIN))
                except Exception:
                    safe_polygon = None
                
                if safe_polygon is None or safe_polygon.is_empty:
                    try:
                        safe_polygon = clean_geom(class_polygon.buffer(-SAFE_MARGIN * 0.5))
                    except Exception:
                        safe_polygon = None
                
                if safe_polygon is None or safe_polygon.is_empty:
                    safe_polygon = class_polygon

            
    
                # ------------------------------------------------------------
                #  FAST NDVI-AWARE SAMPLE GENERATION (NO GRID)
                # ------------------------------------------------------------
                def generate_random_points_fast(polygon, n, oversample=20):
                
                    minx, miny, maxx, maxy = polygon.bounds
                
                    pts = []
                    tries = n * oversample
                
                    xs = np.random.uniform(minx, maxx, tries)
                    ys = np.random.uniform(miny, maxy, tries)
                
                    
                    poly_buffered = polygon.buffer(20)
                    
                    # ✅ simplify BEFORE heavy operations
                    poly_buffered = poly_buffered.simplify(30, preserve_topology=True)
                    
                    # ✅ prepare geometry (VERY IMPORTANT)
                    poly_buffered = prep(poly_buffered)

                    for x, y in zip(xs, ys):
                        p = Point(x, y)
                        if poly_buffered.contains(p):
                            pts.append(p)
                        if len(pts) >= n:
                            break
                
                    return pts
                
                
                def points_to_squares(points, size):
                    half = size / 2
                    return [box(p.x - half, p.y - half, p.x + half, p.y + half) for p in points]
                        
                #  FAST RANDOM POINT GENERATION
                if needed <= 5:
                    factor = 10
                elif needed <= 20:
                    factor = 20
                else:
                    factor = 30
                
                # special case
                if cl == 411:
                    factor *= 4
                
                pts = generate_random_points_fast(safe_polygon, n=needed * factor)
    
                # LIMIT before NDVI (HUGE SPEED BOOST)
                if cl == 411:
                    MAX_GRID = 1200
                else:
                    MAX_GRID = 400
                
                if len(pts) > MAX_GRID:
                    pts = random.sample(pts, MAX_GRID)
                
                if len(pts) == 0:
                    print("  ⚠️ No candidates generated → relaxing constraints")
                
                    # Retry WITHOUT buffer (much less strict)
                    pts = generate_random_points_fast(safe_polygon, n=needed * factor * 2, oversample=50)
                
                    if len(pts) == 0:
                        print("  ❌ Still no candidates — skipping NDVI filter")
                        
                        # fallback: random squares anywhere in bbox
                        minx, miny, maxx, maxy = safe_polygon.bounds
                        pts = [Point(
                            random.uniform(minx, maxx),
                            random.uniform(miny, maxy)
                        ) for _ in range(needed * 5)]
                else:
                    squares = points_to_squares(pts, SQUARE_SIZE * 0.4)
                
                    grid_gdf = gpd.GeoDataFrame(
                        {
                            "clc_l3": [cl] * len(squares),
                            "sample_source": ["generated"] * len(squares),
                        },
                        geometry=squares,
                        crs=sel.crs
                    )
                    
                    # FIX 3 — FILTER VALID GEOMETRIES (VERY IMPORTANT)
                    # STRICT containment (square fully inside class)
                    safe_polygon = clean_geom(safe_polygon)
                    
                    if safe_polygon is None or safe_polygon.is_empty:
                        print("  ⚠️ Invalid safe polygon")
                        grid_gdf = grid_gdf.iloc[0:0].copy()
                    else:
                        safe_prepared = prep(safe_polygon)
                    
                        grid_gdf["geometry"] = grid_gdf.geometry.apply(clean_geom)
                        grid_gdf = grid_gdf[grid_gdf.geometry.notna()].copy()
                    
                        def safe_inside(geom):
                            try:
                                return safe_prepared.contains(geom)
                            except Exception:
                                return False
                    
                        grid_gdf = grid_gdf[grid_gdf.geometry.apply(safe_inside)].copy()

                    
                    if len(grid_gdf) == 0:
                        print("  ⚠️ No valid geometries after intersection filter")
                    else:
                        # NDVI starts here
                        grid_ndvi = grid_gdf.to_crs(ndvi_crs)
                            
                    # ------------------------------------------
                    # 2. COMPUTE NDVI FOR GRID
                    # ------------------------------------------
                    #grid_ndvi = grid_gdf.to_crs(ndvi_crs)
            
                    zs_grid = zonal_stats(
                        grid_ndvi,
                        raster_ndvi_path,
                        stats=["mean"],
                        nodata=ndvi_nodata
                    )
            
                    grid_gdf["ndvi_mean"] = [z["mean"] for z in zs_grid]
            
                    # Remove invalid NDVI
                    grid_gdf = grid_gdf[np.isfinite(grid_gdf["ndvi_mean"])]
            
                    # ------------------------------------------
                    # 3. NDVI TARGETING (CLASS-AWARE)
                    # ------------------------------------------
                    target_ndvi = class_ndvi_target.get(cl, 0)
                    std_ndvi = class_ndvi_std.get(cl, 0.05)
            
                    grid_gdf["ndvi_distance"] = (
                        (grid_gdf["ndvi_mean"] - target_ndvi).abs() / (std_ndvi + 1e-6)
                    )
            
                    # 🔥 Sort by BEST NDVI FIRST
                    grid_gdf = grid_gdf.sort_values("ndvi_distance")
            
                    # ------------------------------------------
                    # 4. GREEDY SELECTION WITH SPACING
                    # ------------------------------------------
                    selected_geoms = list(sel.geometry)  # keep existing best ones
                            
                    MIN_DIST = 40
                    
                    if cl == 411:
                        MIN_DIST = 10
            
                    new_selected = []
                                
                    for geom in grid_gdf.sort_values("ndvi_distance").geometry:
                    
                        if geom is None or geom.is_empty:
                            continue
                    
                        grid_id = get_grid_id(geom, GRID_SIZE)
                    
                        # ❌ skip if used already (VERY IMPORTANT)
                        if grid_id in used_grid_ids:
                            continue
                    
                        c = geom.centroid
                    
                        keep = True
                        for existing in selected_geoms:
                            if c.distance(existing.centroid) < MIN_DIST:
                                keep = False
                                break
                    
                        if keep:
                            new_selected.append(geom)
                            selected_geoms.append(geom)
                            used_grid_ids.add(grid_id)   # ✅ REGISTER HERE
                    
                        if len(new_selected) >= needed:
                            break
                    
                    if len(new_selected) < needed:
                        print(f"  ⚠️ Only filled {len(new_selected)} of {needed}")
                    
                        # OPTIONAL FIX — FORCE FILL FOR HARD CLASSES (e.g. 411)
                        if cl == 411:
                            print("  → Applying fallback fill (relaxed constraints)")
                    
                            for geom in grid_gdf.geometry:
                                if geom not in new_selected:
                                    new_selected.append(geom)
                                if len(new_selected) >= needed:
                                    break      
                                    
                    # ------------------------------------------
                    # 5. MERGE BACK
                    # ------------------------------------------
                    if len(new_selected) > 0:
                        new_gdf = gpd.GeoDataFrame(
                            {
                                "clc_l3": [cl] * len(new_selected),
                                "sample_source": ["generated"] * len(new_selected),
                            },
                            geometry=new_selected,
                            crs=sel.crs
                        )
            
                        sel = pd.concat([sel, new_gdf], ignore_index=True)
            
                        for geom in sel.geometry:
                            used_grid_ids.add(get_grid_id(geom, GRID_SIZE))
                
            final_idx = []
            final_grid_ids = set()
            
            for idx, geom in sel.geometry.items():
            
                if geom is None or geom.is_empty:
                    continue
            
                grid_id = get_grid_id(geom, GRID_SIZE)
            
                if grid_id in final_grid_ids:
                    continue
            
                final_grid_ids.add(grid_id)
                final_idx.append(idx)
            
            sel = sel.loc[final_idx].copy().head(TARGET)
            
            final_rows.append(sel)

            
            print(f"  ⏱️ Took: {time.time() - t_class_start:.2f} sec")

                
        def count_overlaps(gdf):
            gdf = clean_gdf_polygons(gdf)
            s = gdf.sindex
            overlaps = 0
        
            for i, geom in enumerate(gdf.geometry):
                possible = list(s.intersection(geom.bounds))
                for j in possible:
                    if i == j:
                        continue
                    try:
                        if geom.intersects(gdf.geometry.iloc[j]):
                            overlaps += 1
                    except Exception:
                        continue
        
            return overlaps 
            
        final = gpd.GeoDataFrame(pd.concat(final_rows), crs=ranked.crs)
        final["season"] = season_name

        dates_for_season = sorted(set(season_scene_dates.get(season_name, [])))
        final["scene_count"] = len(dates_for_season)
        final["scene_dates"] = "; ".join(dates_for_season)

        print("Final rows:", len(final))
        print("Null clc_l3:", final["clc_l3"].isna().sum())
        print("Overlaps:", count_overlaps(final))

        #print("\nFinal per class (guaranteed 60 each):")
        print("\nFinal per class (non-overlapping):")
        print(final["clc_l3"].value_counts().sort_index())
  
        
        # --------------------------------------------------------------
        # 4.9 SAVE OUTPUTS
        # --------------------------------------------------------------
        final_out = os.path.join(export_dir,f"best_60_per_class_{season_name}.gpkg")
        diag_csv  = os.path.join(export_dir, f"sampling_diagnostics_{season_name}.csv")
        
        final.to_file(final_out, driver="GPKG")
        
        diag = final.groupby("clc_l3").agg(
            n=("clc_l3","count"),
            q25=("quality_score", lambda x: np.percentile(x,25)),
            q50=("quality_score","median"),
            q75=("quality_score", lambda x: np.percentile(x,75)),
            ndvi_std_med=("ndvi_std","median")
        ).reset_index()

        diag["season"] = season_name
        diag["scene_count"] = len(dates_for_season)
        diag["scene_dates"] = "; ".join(dates_for_season)
        
        diag.to_csv(diag_csv, index=False)
        all_season_results.append(final)
        print("Saved:", final_out)
        print("Saved:", diag_csv)

# =========================================================
# MERGE ALL SEASONS
# =========================================================

final_4seasons = gpd.GeoDataFrame(
    pd.concat(all_season_results, ignore_index=True),
    crs=all_season_results[0].crs
)



print("Overlaps:", count_overlaps(final_4seasons))    

print("\nTOTAL SAMPLES:")
print(len(final_4seasons))

print("\nSamples per season:")
print(final_4seasons["season"].value_counts())

print("\nSamples per class per season:")
pivot = (
    final_4seasons
    .groupby(["season", "clc_l3"])
    .size()
    .unstack(fill_value=0)
)

print(pivot)

pivot.to_csv(os.path.join(export_dir, "samples_per_class_per_season.csv"))

merged_out = os.path.join(
    export_dir,
    "best_60_per_class_ALL_SEASONS.gpkg"
)

final_4seasons.to_file(
    merged_out,
    driver="GPKG"
)

print("Saved merged seasonal dataset:")
print(merged_out)

Output()


PROCESSING SEASON: WINTER
Initial candidates: 17715
Reduced candidates: 17715
Candidates CRS: EPSG:3035
NDVI CRS: EPSG:32629


In [ ]:
# 10. FINAL PDF GENERATOR — UPDATED FOR NEW SEASONAL OUTPUTS
# ==========================================================

from rasterio.mask import mask as rio_mask
status_area = widgets.Output()
display(status_area)

with StepTimer("Exporting pdf summary..."):

    output_pdf = os.path.join(export_dir, "pdf", "CLC_class_summary_all_seasons.pdf")
    os.makedirs(os.path.dirname(output_pdf), exist_ok=True)

    # ------------------------------------------------------
    # INPUTS
    # ------------------------------------------------------
    merged_gpkg = os.path.join(export_dir, "best_60_per_class_ALL_SEASONS.gpkg")
    assert os.path.exists(merged_gpkg), "Merged seasonal GPKG not found."

    final_all = gpd.read_file(merged_gpkg)

    # ensure season exists
    if "season" not in final_all.columns:
        raise ValueError("❌ 'season' column not found in merged GPKG.")

    # make sure scene metadata exists (if not, create empty fields)
    if "scene_count" not in final_all.columns:
        final_all["scene_count"] = np.nan

    if "scene_dates" not in final_all.columns:
        final_all["scene_dates"] = ""

    # ------------------------------------------------------
    # LOAD ORIGINAL CLC DATA FOR PARENT POLYGONS
    # ------------------------------------------------------
    gdf_clc_all = gdf_clc_m.copy()

    if gdf_clc_all.crs != final_all.crs:
        gdf_clc_all = gdf_clc_all.to_crs(final_all.crs)

    # unions by class for generated samples
    union_by_class = {
        cl: unary_union(gdf_clc_all.loc[gdf_clc_all["clc_l3"] == cl, "geometry"])
        for cl in gdf_clc_all["clc_l3"].unique()
    }

    # original polygons by class + OBJECTID
    class_objectid_lookup = {}
    if "OBJECTID" in gdf_clc_all.columns:
        for cl, g in gdf_clc_all.groupby("clc_l3"):
            class_objectid_lookup[cl] = g.set_index("OBJECTID")

    # ------------------------------------------------------
    # HELPERS
    # ------------------------------------------------------
    def quality_label(score):
        if score >= 80: return "Excellent"
        elif score >= 60: return "Good"
        elif score >= 40: return "Moderate"
        elif score >= 20: return "Low"
        else: return "Poor"

    def quality_color(score):
        if score >= 80: return "green"
        elif score >= 60: return "yellowgreen"
        elif score >= 40: return "orange"
        elif score >= 20: return "orangered"
        else: return "red"

    def put_border(ax, lw=0.8):
        for s in ax.spines.values():
            s.set_visible(True)
            s.set_linewidth(lw)
            s.set_edgecolor("black")

    def wrap_text(text, width=80):
        return "\n".join(textwrap.wrap(str(text), width))

    def get_polygon_in_crs(poly, orig_crs, target_crs):
        return gpd.GeoSeries([poly], crs=orig_crs).to_crs(target_crs).iloc[0]

    def extract_ndvi_vals(poly_src, poly_crs, ndvi_path):
        with rasterio.open(ndvi_path) as src:
            poly_ndvi = get_polygon_in_crs(poly_src, poly_crs, src.crs)
            arr, _ = rio_mask(src, [mapping(poly_ndvi)], crop=True)
            band = arr[0].astype(float)

            if np.ma.isMaskedArray(band):
                vals = band.compressed()
            else:
                vals = band.ravel()

            vals = vals[np.isfinite(vals)]

            nodata = src.nodata
            if nodata is not None:
                vals = vals[vals != nodata]

            vals = vals[(vals >= -1) & (vals <= 1)]
            return vals

    def extract_ndvi_bbox(poly_src, poly_crs, ndvi_path):
        with rasterio.open(ndvi_path) as src:
            poly_ndvi = get_polygon_in_crs(poly_src, poly_crs, src.crs)
            minx, miny, maxx, maxy = poly_ndvi.bounds

            window = rasterio.windows.from_bounds(minx, miny, maxx, maxy, src.transform)
            nd = src.read(1, window=window).astype(float)

            nodata = src.nodata
            if nodata is not None:
                nd[nd == nodata] = np.nan

            affine = src.window_transform(window)
            return nd, affine, src.crs

    def geom_to_crop_pixels(geom_src, geom_crs, affine, raster_crs):
        g = gpd.GeoSeries([geom_src], crs=geom_crs).to_crs(raster_crs).iloc[0]
        inv = ~affine
        segs = []

        if g.geom_type == "Polygon":
            rings = [g.exterior]
        elif g.geom_type == "MultiPolygon":
            rings = [part.exterior for part in g.geoms]
        else:
            return []

        for ring in rings:
            pts = []
            for x, y in ring.coords:
                col, row = inv * (x, y)
                pts.append((col, row))
            segs.append(pts)

        return segs

    def plot_ndvi_hist(ax, vals):
        vals = vals[np.isfinite(vals)]
        ax.hist(vals, bins=40, color="green", alpha=0.7)
        ax.set_title("NDVI distribution")
        ax.set_xlabel("NDVI")
        ax.set_ylabel("Count")

    def get_parent_polygon(row):
        cl = row["clc_l3"]

        # original sample with matching OBJECTID
        if (
            "sample_source" in row.index
            and row["sample_source"] == "original"
            and "OBJECTID" in row.index
            and pd.notna(row["OBJECTID"])
            and cl in class_objectid_lookup
            and row["OBJECTID"] in class_objectid_lookup[cl].index
        ):
            return class_objectid_lookup[cl].loc[row["OBJECTID"], "geometry"]

        # generated sample fallback: use union polygon of the class
        return union_by_class.get(cl, row.geometry)

    # ------------------------------------------------------
    # PDF GENERATION
    # ------------------------------------------------------
    with PdfPages(output_pdf) as pdf:

        # ============================================
        # PAGE 1 — OVERVIEW
        # ============================================
        fig = plt.figure(figsize=(8.27, 11.69))
        ax = fig.add_subplot(111)
        ax.axis("off")

        txt = (
            "CLC Seasonal Sample Quality Report\n\n"
            "This report summarizes the best samples selected from the merged\n"
            "multi-season dataset.\n\n"
            "For each CLC class, the report shows:\n"
            "• season of the selected best sample\n"
            "• sample source (original / generated)\n"
            "• quality score and NDVI statistics\n"
            "• NDVI map of the parent class polygon\n"
            "• NDVI histogram\n"
            "• acquisition dates used for the seasonal NDVI composite\n"
        )

        ax.text(
            0.03, 0.95,
            txt,
            va="top",
            fontsize=12
        )

        pdf.savefig(fig)
        plt.close(fig)

        # ============================================
        # ONE PAGE PER CLASS — BEST SAMPLE ACROSS ALL SEASONS
        # ============================================
        for cl in sorted(final_all["clc_l3"].dropna().unique()):

            g = final_all[final_all["clc_l3"] == cl].copy()

            if len(g) == 0:
                continue

            g = g.sort_values("quality_score", ascending=False)
            best = g.iloc[0]

            season_name = best["season"]
            ndvi_path = SEASONAL_NDVI[season_name]

            if not os.path.exists(ndvi_path):
                print(f"⚠️ NDVI raster not found for season {season_name}: {ndvi_path}")
                continue

            meta = clc_metadata.get(int(cl), {
                "level1": "Unknown",
                "level2": "Unknown",
                "level3": f"CLC {cl}",
                "description": "No metadata available.",
                "example_folder": None
            })

            parent_poly = get_parent_polygon(best)
            sample_poly = best.geometry

            # Example image
            example_img = None
            folder = meta.get("example_folder", None)
            if folder:
                fp = os.path.join(root_dir, "example_images", folder)
                if os.path.exists(fp):
                    jpgs = [f for f in os.listdir(fp) if f.lower().endswith(".jpg")]
                    if jpgs:
                        import matplotlib.image as mpimg
                        example_img = mpimg.imread(os.path.join(fp, jpgs[0]))

            # NDVI extraction
            ndvi_vals = extract_ndvi_vals(parent_poly, final_all.crs, ndvi_path)
            ndvi_img, affine_ndvi, ndvi_crs = extract_ndvi_bbox(parent_poly, final_all.crs, ndvi_path)

            fig = plt.figure(figsize=(8.27, 11.69), constrained_layout=True)
            gs = gridspec.GridSpec(4, 2, figure=fig, height_ratios=[1.4, 2.6, 2.6, 2.2])

            # -------------------------
            # HEADER
            # -------------------------
            ax_header = fig.add_subplot(gs[0, :])
            ax_header.axis("off")

            title = f"CLC Class {cl} — {meta['level3']}"
            subtitle = f"{meta['level1']} | {meta['level2']}"
            desc = meta["description"]

            ax_header.text(0.01, 0.95, wrap_text(title, 60), fontsize=14, fontweight="bold", va="top")
            ax_header.text(0.01, 0.68, wrap_text(subtitle, 60), fontsize=11, style="italic", va="top")
            ax_header.text(0.01, 0.38, wrap_text(desc, 110), fontsize=10, va="top")

            # -------------------------
            # LEFT: EXAMPLE IMAGE
            # -------------------------
            ax_img = fig.add_subplot(gs[1, 0])
            if example_img is not None:
                ax_img.imshow(example_img)
            else:
                ax_img.text(0.5, 0.5, "No example image available", ha="center", va="center")
            ax_img.axis("off")
            put_border(ax_img)

            # -------------------------
            # RIGHT: STATS
            # -------------------------
            ax_stats = fig.add_subplot(gs[1, 1])
            ax_stats.axis("off")

            q = best["quality_score"]
            label = quality_label(q)
            color = quality_color(q)

            stats_text = f"""
Season: {season_name}
Sample source: {best.get('sample_source', 'unknown')}

Quality score: {q:.1f}
Quality class: {label}

NDVI mean: {best.get('ndvi_mean', np.nan):.3f}
NDVI std: {best.get('ndvi_std', np.nan):.3f}
NDVI range: {best.get('ndvi_range', np.nan):.3f}

Interior margin: {best.get('interior_margin_m', np.nan):.1f} m

Scene count: {best.get('scene_count', np.nan)}
Scene dates:
{wrap_text(best.get('scene_dates', ''), 42)}
            """.strip()

            ax_stats.text(
                0.02, 0.98,
                "Sample statistics",
                fontsize=11,
                fontweight="bold",
                va="top"
            )

            ax_stats.text(
                0.02, 0.88,
                stats_text,
                fontsize=10,
                va="top",
                color="black"
            )
            put_border(ax_stats)

            # -------------------------
            # NDVI MAP
            # -------------------------
            gs_maps = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[2, :], wspace=0.15)

            ax_ndvi = fig.add_subplot(gs_maps[0, 0])
            ax_ndvi.imshow(ndvi_img, cmap="RdYlGn")
            ax_ndvi.set_title(f"Seasonal NDVI ({season_name})", fontsize=11)
            ax_ndvi.axis("off")

            for seg in geom_to_crop_pixels(parent_poly, final_all.crs, affine_ndvi, ndvi_crs):
                xs, ys = zip(*seg)
                ax_ndvi.plot(xs, ys, "-r", lw=1.5)

            for seg in geom_to_crop_pixels(sample_poly, final_all.crs, affine_ndvi, ndvi_crs):
                xs, ys = zip(*seg)
                ax_ndvi.plot(xs, ys, "-y", lw=1.5)

            put_border(ax_ndvi)

            # -------------------------
            # RIGHT MAP PANEL:
            # class overview text instead of RGB
            # -------------------------
            ax_txt = fig.add_subplot(gs_maps[0, 1])
            ax_txt.axis("off")
            ax_txt.text(
                0.02, 0.95,
                "Interpretation",
                fontsize=11,
                fontweight="bold",
                va="top"
            )
            ax_txt.text(
                0.02, 0.80,
                "Red outline = original class polygon\n"
                "Yellow outline = selected sample\n\n"
                "This NDVI map corresponds to the season\n"
                "from which the selected sample was ranked.\n\n"
                "The scene dates shown in the statistics panel\n"
                "are the Sentinel-2 acquisitions used to build\n"
                "the seasonal NDVI composite.",
                fontsize=10,
                va="top"
            )
            put_border(ax_txt)

            # -------------------------
            # HISTOGRAM
            # -------------------------
            ax_hist = fig.add_subplot(gs[3, :])
            if len(ndvi_vals) > 0:
                plot_ndvi_hist(ax_hist, ndvi_vals)
            else:
                ax_hist.text(0.5, 0.5, "No NDVI values available", ha="center", va="center")
            put_border(ax_hist)

            pdf.savefig(fig)
            plt.close(fig)

    print("Summary PDF created:", output_pdf)